<a href="https://colab.research.google.com/github/ysur0004-yuwan/MMA3001---PROJECT/blob/main/PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("API"))
project = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging")
version = project.version(4)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.2/361.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pork-Rasher-Error(Packaging)-4 in yolov8:: 100%|██████████| 7103/7103 [00:01<00:00, 6729.45it/s]


In [ ]:
import os
from pathlib import Path
from collections import Counter

dataset_path = Path(dataset.location)  # path Roboflow downloaded to

# 1. Count images per split
for split in ["train", "valid", "test"]:
    img_dir = dataset_path / split / "images"
    if img_dir.exists():
        n_images = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        print(f"{split}: {n_images} images")

# 2. Count label instances per class, and flag multi-defect images
class_counts = Counter()
multi_defect_images = 0
pass_images = 0
fail_images = 0

for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"
    if not label_dir.exists():
        continue
    for label_file in label_dir.glob("*.txt"):
        with open(label_file) as f:
            lines = [l.strip() for l in f if l.strip()]
        if len(lines) == 0:
            pass_images += 1
            continue
        fail_images += 1
        classes_in_image = set(line.split()[0] for line in lines)
        if len(classes_in_image) > 1:
            multi_defect_images += 1
        for line in lines:
            class_id = line.split()[0]
            class_counts[class_id] += 1

print("\nClass label counts (by class ID):", dict(class_counts))
print(f"\nPass images (no defect): {pass_images}")
print(f"Fail images (>=1 defect): {fail_images}")
print(f"Images with multiple defect types: {multi_defect_images}")

# 3. Check class ID -> class name mapping (from data.yaml)
import yaml
with open(dataset_path / "data.yaml") as f:
    data_yaml = yaml.safe_load(f)
print("\nClass names:", data_yaml.get("names"))

train: 3349 images
valid: 120 images
test: 80 images

Class label counts (by class ID): {'1': 1769, '3': 1185, '2': 79, '4': 18, '0': 72}

Pass images (no defect): 710
Fail images (>=1 defect): 2839
Images with multiple defect types: 91

Class names: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']


In [ ]:
from collections import defaultdict

# Nested counter: per_split_class_counts[split][class_id] = count
# defaultdict(Counter) means every new split automatically starts with an empty Counter
per_split_class_counts = defaultdict(lambda: Counter())

# Loop through each predefined dataset split
for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"

    # Skip if this split's label folder doesn't exist for some reason
    if not label_dir.exists():
        continue

    # Each .txt file corresponds to one image's YOLO-format annotations
    for label_file in label_dir.glob("*.txt"):
        with open(label_file) as f:
            # One line per bounding box annotation in this image; strip blank lines
            lines = [l.strip() for l in f if l.strip()]

        # YOLO format: each line starts with the class ID, e.g. "3 0.45 0.52 0.12 0.08"
        for line in lines:
            class_id = line.split()[0]
            per_split_class_counts[split][class_id] += 1

# Map class IDs (strings "0"-"4") back to their human-readable names
class_names = data_yaml.get("names")

# Print a clean per-split, per-class breakdown
for split in ["train", "valid", "test"]:
    print(f"\n{split}:")
    for class_id, name in enumerate(class_names):
        # .get(..., 0) so classes with zero instances in a split still print as 0, not crash
        print(f"  {name}: {per_split_class_counts[split].get(str(class_id), 0)}")


train:
  loose-meat: 66
  packaging-error: 1712
  twisted-meat: 72
  unsealed: 1055
  wrinkle: 18

valid:
  loose-meat: 3
  packaging-error: 34
  twisted-meat: 3
  unsealed: 81
  wrinkle: 0

test:
  loose-meat: 3
  packaging-error: 23
  twisted-meat: 4
  unsealed: 49
  wrinkle: 0


In [ ]:
# Import the module (upload data_utils.py to the Colab session first)
from pathlib import Path
import data_utils as du

dataset_path = Path(dataset.location)  # from your earlier Roboflow download

class_names = du.load_class_names(dataset_path)
labels_df = du.build_image_level_labels(dataset_path, class_names)
du.summarise_labels(labels_df)

Images per split:
split
train    3349
valid     120
test       80
Name: count, dtype: int64

Pass/fail counts (overall):
pass_fail
fail    2839
pass     710
Name: count, dtype: int64

Extension label counts (excludes pass and multi-defect images):
extension_label
packaging-error    1498
unsealed           1116
twisted-meat         64
loose-meat           52
wrinkle              18
Name: count, dtype: int64

Multi-defect images excluded from extension task: 91


In [ ]:
# Build train/validation/test datasets for the pass/fail task
train_ds = du.make_pass_fail_dataset(labels_df, split="train", shuffle=True)
valid_ds = du.make_pass_fail_dataset(labels_df, split="valid", shuffle=False)
test_ds = du.make_pass_fail_dataset(labels_df, split="test", shuffle=False)

# Import the new module
import model_utils as mu

# Compute class weights to correct for the 80/20 pass/fail imbalance
class_weights = mu.compute_class_weights(labels_df, split="train")
print("Class weights:", class_weights)

# Build and train the baseline
model = mu.build_baseline_model()
history = mu.train_baseline_model(model, train_ds, valid_ds, class_weight=class_weights, epochs=15)

Class weights: {0: np.float64(2.448099415204678), 1: np.float64(0.6283302063789868)}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 169ms/step - accuracy: 0.4377 - loss: 0.7820 - precision: 0.8069 - recall: 0.3857 - val_accuracy: 0.5083 - val_loss: 0.6791 - val_precision: 0.8406 - val_recall: 0.5472
Epoch 2/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.5494 - loss: 0.7115 - precision: 0.8255 - recall: 0.5501 - val_accuracy: 0.5833 - val_loss: 0.6559 - val_precision: 0.9000 - val_recall: 0.5943
Epoch 3/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.5766 - loss: 0.6874 - precision: 0.8458 - recall: 0.5722 - val_accuracy: 0.6917 - val_loss: 0.5920 - val_precision: 0.9157 - val_recall: 0.7170
Epoch 4/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.5829 - loss: 0.6674 - precision: 0.8590 - recall: 0.5692 - val_accuracy: 0.8417 - val_loss: 0.4663 - val_precision: 0.9065 - val_recall: 0.9151
Epoch 5/15
105/10

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_pass_fail_model(model, dataset, class_names=("pass", "fail"), threshold=0.5):
    y_true = []
    y_pred_probs = []

    for images, labels in dataset:
        y_true.extend(labels.numpy())
        batch_probs = model.predict(images, verbose=0)
        y_pred_probs.extend(batch_probs.flatten())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)
    y_pred = (y_pred_probs >= threshold).astype(int)

    report = classification_report(y_true, y_pred, target_names=list(class_names), zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print("Classification report (test set):")
    print(report)
    print("Confusion matrix (rows = true, columns = predicted):")
    print(f"               pred_{class_names[0]}   pred_{class_names[1]}")
    print(f"true_{class_names[0]:<8}  {cm[0][0]:>10}  {cm[0][1]:>10}")
    print(f"true_{class_names[1]:<8}  {cm[1][0]:>10}  {cm[1][1]:>10}")

    return {"y_true": y_true, "y_pred": y_pred, "y_pred_probs": y_pred_probs, "confusion_matrix": cm}

results = evaluate_pass_fail_model(model, test_ds, class_names=("pass", "fail"))

Classification report (test set):
              precision    recall  f1-score   support

        pass       0.41      0.75      0.53        12
        fail       0.95      0.81      0.87        68

    accuracy                           0.80        80
   macro avg       0.68      0.78      0.70        80
weighted avg       0.87      0.80      0.82        80

Confusion matrix (rows = true, columns = predicted):
               pred_pass   pred_fail
true_pass               9           3
true_fail              13          55


In [ ]:
# Check precision/recall trade-off at different thresholds, instead of the
# default 0.5. Lower thresholds mean "fail" is predicted more readily,
# which should catch more real defects (higher recall) at the cost of
# more false alarms (lower precision) — a trade-off worth making here,
# since a missed defect skips human review entirely, while a false alarm
# just costs a human a quick look.
from sklearn.metrics import classification_report

y_true = results["y_true"]
y_pred_probs = results["y_pred_probs"]

for threshold in [0.5, 0.4, 0.3, 0.2]:
    y_pred_at_t = (y_pred_probs >= threshold).astype(int)
    print(f"\n--- Threshold = {threshold} ---")
    print(classification_report(y_true, y_pred_at_t, target_names=["pass", "fail"], zero_division=0))


--- Threshold = 0.5 ---
              precision    recall  f1-score   support

        pass       0.41      0.75      0.53        12
        fail       0.95      0.81      0.87        68

    accuracy                           0.80        80
   macro avg       0.68      0.78      0.70        80
weighted avg       0.87      0.80      0.82        80


--- Threshold = 0.4 ---
              precision    recall  f1-score   support

        pass       0.46      0.50      0.48        12
        fail       0.91      0.90      0.90        68

    accuracy                           0.84        80
   macro avg       0.69      0.70      0.69        80
weighted avg       0.84      0.84      0.84        80


--- Threshold = 0.3 ---
              precision    recall  f1-score   support

        pass       0.75      0.25      0.38        12
        fail       0.88      0.99      0.93        68

    accuracy                           0.88        80
   macro avg       0.82      0.62      0.65        80

In [ ]:
# Builds a tf.data.Dataset for the defect-type extension task.
# Only images with a single, unambiguous defect type are included —
# pass images and the 91 multi-defect images are excluded (extension_label
# is None for both), per the policy documented in Section 2.
import tensorflow as tf
def make_defect_type_dataset(df, split, class_names, shuffle=True):
    split_df = df[(df["split"] == split) & (df["extension_label"].notna())].copy()

    # Map each class name to a fixed integer index, consistent across splits
    label_to_index = {name: i for i, name in enumerate(class_names)}
    split_df["label_idx"] = split_df["extension_label"].map(label_to_index)

    paths = split_df["image_path"].tolist()
    labels = split_df["label_idx"].tolist()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load_and_preprocess(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, (224, 224))
        image = image / 255.0
        return image, label

    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=max(len(paths), 1))
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

# Build the three splits for the extension task
defect_train_ds = make_defect_type_dataset(labels_df, "train", class_names, shuffle=True)
defect_valid_ds = make_defect_type_dataset(labels_df, "valid", class_names, shuffle=False)
defect_test_ds = make_defect_type_dataset(labels_df, "test", class_names, shuffle=False)

print("Classes (in index order):", class_names)

Classes (in index order): ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']


In [ ]:
import numpy as np

# Multi-class version of the same transfer-learning approach as the baseline:
# frozen MobileNetV2 backbone + a small trainable head, now with 5 softmax outputs.
def build_defect_type_model(num_classes, input_shape=(224, 224, 3), learning_rate=1e-4):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape, include_top=False, weights="imagenet"
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="defect_type_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

# Class weights to correct for the severe imbalance (e.g. packaging-error:
# 1,498 vs wrinkle: 18 in image-level counts, per Section 2's data audit)
def compute_multiclass_class_weights(df, class_names, split="train"):
    split_df = df[(df["split"] == split) & (df["extension_label"].notna())]
    counts = split_df["extension_label"].value_counts()
    total = counts.sum()
    n_classes = len(class_names)
    weights = {}
    for i, name in enumerate(class_names):
        count = counts.get(name, 0)
        weights[i] = total / (n_classes * count) if count > 0 else 0.0
    return weights

defect_class_weights = compute_multiclass_class_weights(labels_df, class_names, split="train")
print("Class weights:", {class_names[i]: round(w, 2) for i, w in defect_class_weights.items()})

defect_model = build_defect_type_model(num_classes=len(class_names))

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("defect_type_model.keras", monitor="val_loss", save_best_only=True),
]

defect_history = defect_model.fit(
    defect_train_ds,
    validation_data=defect_valid_ds,
    epochs=15,
    class_weight=defect_class_weights,
    callbacks=callbacks,
)

Class weights: {'loose-meat': np.float64(10.75), 'packaging-error': np.float64(0.35), 'twisted-meat': np.float64(9.06), 'unsealed': np.float64(0.52), 'wrinkle': np.float64(28.68)}
Epoch 1/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 29s 184ms/step - accuracy: 0.2813 - loss: 2.0408 - val_accuracy: 0.2255 - val_loss: 1.6408
Epoch 2/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.2770 - loss: 1.7191 - val_accuracy: 0.3333 - val_loss: 1.5320
Epoch 3/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.3134 - loss: 1.6540 - val_accuracy: 0.4706 - val_loss: 1.3748
Epoch 4/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.3588 - loss: 1.5659 - val_accuracy: 0.5098 - val_loss: 1.3257
Epoch 5/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.4200 - loss: 1.4790 - val_accuracy: 0.5784 - val_loss: 1.2684
Epoch 6/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.4436 - loss: 1.3624 - val_accuracy: 0.6078 - val_loss: 1.1442
Epoch 7/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - a

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Evaluates the defect-type model on a held-out dataset (should be the test
# split). labels=range(len(class_names)) is passed explicitly so that
# wrinkle still appears as a row/column even though it has zero test
# examples, rather than silently vanishing from the report.
def evaluate_defect_type_model(model, dataset, class_names):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        y_true.extend(labels.numpy())
        probs = model.predict(images, verbose=0)
        preds = np.argmax(probs, axis=1)  # pick the highest-probability class
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    report = classification_report(
        y_true, y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,  # avoids crashing/NaN on wrinkle's zero support
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    print("Classification report (test set):")
    print(report)
    print(f"Confusion matrix (rows=true, cols=pred), class order: {class_names}")
    print(cm)

    return {"y_true": y_true, "y_pred": y_pred, "confusion_matrix": cm}

defect_results = evaluate_defect_type_model(defect_model, defect_test_ds, class_names)

Classification report (test set):
                 precision    recall  f1-score   support

     loose-meat       0.00      0.00      0.00         2
packaging-error       0.45      0.38      0.42        13
   twisted-meat       0.00      0.00      0.00         4
       unsealed       0.93      0.91      0.92        46
        wrinkle       0.00      0.00      0.00         0

       accuracy                           0.72        65
      macro avg       0.28      0.26      0.27        65
   weighted avg       0.75      0.72      0.74        65

Confusion matrix (rows=true, cols=pred), class order: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']
[[ 0  1  0  0  1]
 [ 0  5  2  3  3]
 [ 1  3  0  0  0]
 [ 1  2  0 42  1]
 [ 0  0  0  0  0]]


In [ ]:
def build_simple_cnn_model(input_shape=(224, 224, 3), learning_rate=1e-4):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=input_shape),
        tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name="simple_cnn_from_scratch")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
    )
    return model

simple_cnn_model = build_simple_cnn_model()
simple_cnn_model.summary()

print(f"\nMobileNetV2 baseline total params: {model.count_params():,}")
print(f"Simple CNN total params: {simple_cnn_model.count_params():,}")

import time

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("simple_cnn_model.keras", monitor="val_loss", save_best_only=True),
]

start_time = time.time()
simple_cnn_history = simple_cnn_model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
)
elapsed = time.time() - start_time
print(f"\nTotal training time: {elapsed:.1f} seconds")

Model: "simple_cnn_from_scratch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,809 (108.63 KB)

 Trainable params: 27,809 (108.63 KB)

 Non-trainable params: 0 (0.00 B)


MobileNetV2 baseline total params: 2,259,265
Simple CNN total params: 27,809
Epoch 1/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 30s 115ms/step - accuracy: 0.2738 - loss: 0.6930 - precision: 0.8397 - recall: 0.1081 - val_accuracy: 0.1167 - val_loss: 0.6949 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - accuracy: 0.6450 - loss: 0.6936 - precision: 0.7924 - recall: 0.7505 - val_accuracy: 0.8833 - val_loss: 0.6901 - val_precision: 0.8833 - val_recall: 1.0000
Epoch 3/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 11s 39ms/step - accuracy: 0.5724 - loss: 0.6935 - precision: 0.7882 - recall: 0.6326 - val_accuracy: 0.8833 - val_loss: 0.6921 - val_precision: 0.8833 - val_recall: 1.0000
Epoch 4/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step - accuracy: 0.5873 - loss: 0.6934 - precision: 0.7977 - recall: 0.6450 - val_accuracy: 0.8833 - val_loss: 0.6897 - val_precision: 0.8833 - val_recall: 1.0000
Epoch 5/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 12s 40ms/step - accuracy:

In [ ]:
simple_cnn_results = evaluate_pass_fail_model(simple_cnn_model, test_ds, class_names=("pass", "fail"))

Classification report (test set):
              precision    recall  f1-score   support

        pass       0.00      0.00      0.00        12
        fail       0.85      1.00      0.92        68

    accuracy                           0.85        80
   macro avg       0.42      0.50      0.46        80
weighted avg       0.72      0.85      0.78        80

Confusion matrix (rows = true, columns = predicted):
               pred_pass   pred_fail
true_pass               0          12
true_fail               0          68
